In [1]:
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from torch import autograd
import time

# LeNet Implementation in PyTorch

This notebook implements the LeNet-5 convolutional neural network for handwritten digit recognition using MNIST dataset.

In [2]:
# Check for MPS (Apple Silicon GPU) availability
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5, padding=2)
        self.conv3 = nn.Conv2d(16, 120, 5)
        self.pool3 = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3(x)))
        x = x.view(-1, 120)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [4]:
import torchvision
import torchvision.transforms as transforms

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load MNIST dataset
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [5]:
# Initialize model, loss, and optimizer
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Training finished')

Epoch 1, Batch 100, Loss: 2.298
Epoch 1, Batch 200, Loss: 2.249
Epoch 1, Batch 300, Loss: 1.436
Epoch 1, Batch 400, Loss: 0.718
Epoch 1, Batch 500, Loss: 0.420
Epoch 1, Batch 600, Loss: 0.276
Epoch 1, Batch 700, Loss: 0.234
Epoch 1, Batch 800, Loss: 0.202
Epoch 1, Batch 900, Loss: 0.180
Epoch 2, Batch 100, Loss: 0.144
Epoch 2, Batch 200, Loss: 0.138
Epoch 2, Batch 300, Loss: 0.130
Epoch 2, Batch 400, Loss: 0.138
Epoch 2, Batch 500, Loss: 0.112
Epoch 2, Batch 600, Loss: 0.110
Epoch 2, Batch 700, Loss: 0.111
Epoch 2, Batch 800, Loss: 0.115
Epoch 2, Batch 900, Loss: 0.102
Epoch 3, Batch 100, Loss: 0.091
Epoch 3, Batch 200, Loss: 0.093
Epoch 3, Batch 300, Loss: 0.086
Epoch 3, Batch 400, Loss: 0.091
Epoch 3, Batch 500, Loss: 0.077
Epoch 3, Batch 600, Loss: 0.084
Epoch 3, Batch 700, Loss: 0.084
Epoch 3, Batch 800, Loss: 0.082
Epoch 3, Batch 900, Loss: 0.074
Epoch 4, Batch 100, Loss: 0.071
Epoch 4, Batch 200, Loss: 0.073
Epoch 4, Batch 300, Loss: 0.066
Epoch 4, Batch 400, Loss: 0.075
Epoch 4,

In [6]:
# Move model to device
model = model.to(device)

# Test the model
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on test set: {100 * correct / total:.2f}%')

Accuracy on test set: 98.46%
